# Entraînement distribué avec DDP

Dans ce TP, nous allons répartir l'entraînement d'un modèle sur plusieurs processus avec `DistributedDataParallel` (DDP), le mécanisme standard de PyTorch pour le **parallélisme de données**.

Colab ne fournit qu'un seul GPU : nous lancerons donc 2 processus **sur CPU**, qui communiquent avec le backend `gloo`. Sur une machine à 4 ou 8 GPUs, le même code ne change qu'en deux points : le backend (`nccl`), et le placement du modèle sur le GPU de chaque processus (son numéro sur la machine, `LOCAL_RANK`, passé à `torch.cuda.set_device`, à `.to()` et aux `device_ids` de `DDP`).

Au programme :

1. écrire un script d'entraînement DDP et le lancer avec `torchrun` ;
2. vérifier que les processus restent synchronisés ;
3. sauvegarder un checkpoint correctement ;
4. faire la même chose en quelques lignes avec PyTorch Lightning ;
5. en option, entraîner en précision mixte.

In [ ]:
!pip install -q lightning

## Le problème de départ

Voici un script d'entraînement classique, sur un seul processus, pour un problème de classification synthétique. Le cadre `%%writefile` écrit le contenu de la cellule dans un fichier au lieu de l'exécuter.

In [ ]:
%%writefile commun.py
import torch
from torch import nn
from torch.utils.data import TensorDataset


def fabrique_donnees(n: int = 8192) -> TensorDataset:
  # Même graine dans tous les processus : tous voient le même jeu de données
  generateur = torch.Generator().manual_seed(0)
  x = torch.randn(n, 20, generator=generateur)
  vrais_poids = torch.randn(20, 3, generator=generateur)
  y = (x @ vrais_poids).argmax(dim=1)
  return TensorDataset(x, y)


def fabrique_modele() -> nn.Module:
  torch.manual_seed(0)
  return nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 3))

In [ ]:
%%writefile entrainement_simple.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from commun import fabrique_donnees, fabrique_modele

donnees = fabrique_donnees()
modele = fabrique_modele()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
chargeur = DataLoader(donnees, batch_size=64, shuffle=True)

for epoque in range(3):
  for x, y in chargeur:
    optimiseur.zero_grad()
    perte = cross_entropy(modele(x), y)
    perte.backward()
    optimiseur.step()
  print(f"Epoch {epoque} : {len(chargeur)} batchs, dernière perte {perte.item():.3f}")

In [ ]:
!python entrainement_simple.py

## Un script DDP

Avec DDP, `torchrun` lance plusieurs **processus** qui exécutent le même script. Chaque processus :

- rejoint le groupe de processus (`dist.init_process_group`) ; `torchrun` lui fournit son numéro (*rang*) et le nombre total de processus dans des variables d'environnement ;
- enveloppe son modèle dans `DistributedDataParallel`, qui moyenne les gradients entre processus pendant `backward()` ;
- ne lit qu'une part du jeu de données, grâce à un `DistributedSampler`.

*Complétez le script `entrainement_ddp.py` ci-dessous (les `...`) :*

- *initialisez le groupe de processus avec le backend `"gloo"` et récupérez le rang et le nombre de processus (`dist.get_rank()`, `dist.get_world_size()`) ;*
- *enveloppez le modèle dans `DDP` ;*
- *créez un `DistributedSampler` et passez-le au `DataLoader` (à la place de `shuffle=True`) ;*
- *appelez `echantillonneur.set_epoch(epoque)` au début de chaque epoch ;*
- *n'affichez les messages que depuis le processus de rang 0.*

In [ ]:
%%writefile entrainement_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from commun import fabrique_donnees, fabrique_modele

...  # Initialisation du groupe de processus, rang et nombre de processus

donnees = fabrique_donnees()
modele = ...  # Modèle enveloppé dans DDP
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = ...
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  ...
  for x, y in chargeur:
    optimiseur.zero_grad()
    perte = cross_entropy(modele(x), y)
    perte.backward()
    optimiseur.step()
  ...  # Affichage depuis le rang 0 seulement

dist.destroy_process_group()

### Conseils supplémentaires

- `dist.init_process_group("gloo")` s'appelle avant toute autre fonction de `dist` : `dist.get_rank()` et `dist.get_world_size()` en dépendent.
- Sur CPU, `DDP` ne prend que le modèle : `DDP(fabrique_modele())`, sans `device_ids`.
- Le `DistributedSampler` mélange lui-même les données (`shuffle=True` par défaut) : le `DataLoader` refuse `shuffle=True` avec un `sampler`.
- `set_epoch(epoque)` s'appelle avant la boucle sur les batchs, pour que chaque epoch ait son propre mélange.

```python
dist.init_process_group(...)
rang = ...
nb_processus = ...

donnees = fabrique_donnees()
modele = DDP(...)
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = DistributedSampler(...)
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  echantillonneur.set_epoch(...)
  for x, y in chargeur:
    optimiseur.zero_grad()
    perte = cross_entropy(modele(x), y)
    perte.backward()
    optimiseur.step()
  if ...:
    print(f"Epoch {epoque} : dernière perte {perte.item():.3f}")
```

### Solution

In [ ]:
%%writefile entrainement_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from commun import fabrique_donnees, fabrique_modele

# Sur GPU, d'abord le GPU de ce processus, son numéro sur la machine (LOCAL_RANK) :
# rang_local = dist.get_node_local_rank(); torch.cuda.set_device(rang_local)
dist.init_process_group("gloo")  # "nccl" sur GPU
rang = dist.get_rank()
nb_processus = dist.get_world_size()

donnees = fabrique_donnees()
modele = DDP(fabrique_modele())  # Sur GPU : .to(rang_local), device_ids=[rang_local]
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = DistributedSampler(donnees)
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  echantillonneur.set_epoch(epoque)
  for x, y in chargeur:
    optimiseur.zero_grad()
    perte = cross_entropy(modele(x), y)
    perte.backward()
    optimiseur.step()
  if rang == 0:
    print(
      f"Epoch {epoque} : {len(chargeur)} batchs par processus, "
      f"{nb_processus} processus, dernière perte {perte.item():.3f}"
    )

dist.destroy_process_group()

### Lancer le script

Lancez le script sur 2 processus :

In [ ]:
!torchrun --nproc_per_node=2 entrainement_ddp.py

## Analyse de l'entraînement parallèle

1. *Combien de batchs chaque processus traite-t-il par epoch, comparé au script sur un seul processus ? Pourquoi ?*
2. *Chaque processus utilise des batchs de 64 exemples. Combien d'exemples contribuent à chaque mise à jour des poids ? Quelle conséquence sur le choix du learning rate ?*
3. *À quoi sert `echantillonneur.set_epoch(epoque)` ?*

*Vos réponses ici.*

### Solution

1. Chaque processus traite 64 batchs par epoch au lieu de 128 : le `DistributedSampler` donne à chaque processus une moitié différente du jeu de données. Avec 2 GPUs, une epoch prend donc environ deux fois moins de temps. Ici, sur CPU, le script DDP est au contraire plus lent que celui à un seul processus : le lancement des processus, l'échange des gradients par `gloo` à chaque batch et un seul thread par processus (`torchrun` fixe `OMP_NUM_THREADS=1`) coûtent plus que ce que le partage des données fait gagner sur un si petit modèle.
2. Les gradients sont moyennés entre les 2 processus : chaque mise à jour utilise 2 × 64 = 128 exemples. Le batch effectif grandit avec le nombre de processus ; on augmente souvent le learning rate en proportion (avec un *warmup*), sinon l'entraînement avance moins par epoch.
3. Le `DistributedSampler` mélange les données avec une graine qui dépend de l'epoch : sans `set_epoch`, chaque epoch verrait les exemples dans le même ordre.

## Les processus restent-ils synchronisés ?

Chaque processus a sa propre copie du modèle. DDP garantit qu'elles restent identiques : les poids sont diffusés depuis le rang 0 à la création du `DDP`, puis tous les processus appliquent la même mise à jour, calculée à partir des gradients moyennés.

*Vérifions-le. Le script `entrainement_ddp_verif.py` ci-dessous reprend l'entraînement ; complétez sa fin (les `...`) : calculez dans chaque processus la somme de tous les poids du modèle, rassemblez ces sommes avec `dist.all_gather`, puis affichez-les depuis le rang 0.*

**Documentation utile**

- [`torch.distributed.all_gather`](https://docs.pytorch.org/docs/stable/distributed.html#torch.distributed.all_gather)

In [ ]:
%%writefile entrainement_ddp_verif.py
import sys

import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from commun import fabrique_donnees, fabrique_modele

# Lancé avec --sans-ddp, le script entraîne le modèle sans l'envelopper dans DDP
avec_ddp = "--sans-ddp" not in sys.argv

dist.init_process_group("gloo")
rang = dist.get_rank()
nb_processus = dist.get_world_size()

donnees = fabrique_donnees()
modele = DDP(fabrique_modele()) if avec_ddp else fabrique_modele()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = DistributedSampler(donnees)
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  echantillonneur.set_epoch(epoque)
  for x, y in chargeur:
    optimiseur.zero_grad()
    cross_entropy(modele(x), y).backward()
    optimiseur.step()

somme_controle = ...  # Somme de tous les poids du modèle, dans ce processus
sommes_controle = ...  # La liste que dist.all_gather remplit, une place par processus
...  # Rassemblement des sommes de tous les processus
if rang == 0:
  print("Somme des poids dans chaque processus :")
  print([f"{s.item():.6f}" for s in sommes_controle])

dist.destroy_process_group()

### Conseils supplémentaires

- `p.detach().sum()` : la somme des éléments d'un tenseur `p` de `modele.parameters()`, hors du graphe de calcul ; `torch.stack(...)` réunit ces sommes en un tenseur, dont `.sum()` donne la somme de tous les poids, un tenseur à zéro dimension.
- `dist.all_gather(liste, tenseur)` remplit `liste`, qui contient déjà un tenseur de la même forme par processus : `[torch.zeros(()) for _ in range(nb_processus)]`.
- Tous les processus appellent `dist.all_gather`, pas seulement le rang 0 : c'est une opération collective, qui attend chacun d'eux.

```python
somme_controle = torch.stack([... for p in modele.parameters()]).sum()
sommes_controle = [torch.zeros(()) for _ in range(...)]
dist.all_gather(..., ...)
```

### Solution

In [ ]:
%%writefile entrainement_ddp_verif.py
import sys

import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from commun import fabrique_donnees, fabrique_modele

# Lancé avec --sans-ddp, le script entraîne le modèle sans l'envelopper dans DDP
avec_ddp = "--sans-ddp" not in sys.argv

dist.init_process_group("gloo")
rang = dist.get_rank()
nb_processus = dist.get_world_size()

donnees = fabrique_donnees()
modele = DDP(fabrique_modele()) if avec_ddp else fabrique_modele()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = DistributedSampler(donnees)
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  echantillonneur.set_epoch(epoque)
  for x, y in chargeur:
    optimiseur.zero_grad()
    cross_entropy(modele(x), y).backward()
    optimiseur.step()

somme_controle = torch.stack([p.detach().sum() for p in modele.parameters()]).sum()
sommes_controle = [torch.zeros(()) for _ in range(nb_processus)]
dist.all_gather(sommes_controle, somme_controle)
if rang == 0:
  print("Somme des poids dans chaque processus :")
  print([f"{s.item():.6f}" for s in sommes_controle])

dist.destroy_process_group()

### Avec et sans DDP

Lancez le script sur 2 processus, puis relancez-le avec `--sans-ddp` : le modèle n'est plus enveloppé dans `DDP`, et chaque processus l'entraîne seul.

*Les sommes sont-elles égales dans les deux cas ? Pourquoi ?*

In [ ]:
!torchrun --nproc_per_node=2 entrainement_ddp_verif.py

In [ ]:
!torchrun --nproc_per_node=2 entrainement_ddp_verif.py --sans-ddp

*Vos réponses ici.*

#### Solution

Avec DDP, les deux sommes sont identiques : les copies partent des mêmes poids, diffusés depuis le rang 0, et reçoivent toutes la même mise à jour, calculée à partir des gradients moyennés. Sans DDP, elles diffèrent : les deux processus partent bien des mêmes poids (`fabrique_modele` fixe la graine), mais chacun ne voit que sa moitié des données et applique ses propres gradients, sans les moyenner avec ceux de l'autre : les copies divergent dès la première mise à jour.

## Sauvegarder un checkpoint

Dans un entraînement distribué, tous les processus ont les mêmes poids : un seul doit écrire le checkpoint, sinon ils écrivent tous le même fichier en même temps.

*Modifiez `entrainement_ddp.py` pour sauvegarder le modèle à la fin de l'entraînement, depuis le rang 0 seulement, dans `modele.pt`. Vérifiez ensuite, dans une cellule du notebook, que vous pouvez recharger ces poids dans un modèle **non distribué** créé avec `fabrique_modele()`.*

*Indice : le modèle d'origine est accessible dans `modele.module`. Pourquoi faut-il sauvegarder `modele.module.state_dict()` plutôt que `modele.state_dict()` ?*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `modele.module` : le modèle d'origine, sans l'enveloppe `DDP`.
- La sauvegarde vient après la boucle d'entraînement, dans un `if rang == 0:`, avant `dist.destroy_process_group()`.
- Dans le notebook, `load_state_dict` prend le dictionnaire que `torch.load` relit.

```python
if rang == 0:  # Dans entrainement_ddp.py, après la boucle d'entraînement
  torch.save(..., "modele.pt")

modele_recharge = fabrique_modele()  # Dans une cellule du notebook
modele_recharge.load_state_dict(...)
```

### Solution

In [ ]:
%%writefile entrainement_ddp.py
import torch
import torch.distributed as dist
from torch.nn.functional import cross_entropy
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

from commun import fabrique_donnees, fabrique_modele

dist.init_process_group("gloo")
rang = dist.get_rank()
nb_processus = dist.get_world_size()

donnees = fabrique_donnees()
modele = DDP(fabrique_modele())
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
echantillonneur = DistributedSampler(donnees)
chargeur = DataLoader(donnees, batch_size=64, sampler=echantillonneur)

for epoque in range(3):
  echantillonneur.set_epoch(epoque)
  for x, y in chargeur:
    optimiseur.zero_grad()
    perte = cross_entropy(modele(x), y)
    perte.backward()
    optimiseur.step()
  if rang == 0:
    print(f"Epoch {epoque} : dernière perte {perte.item():.3f}")

if rang == 0:
  torch.save(modele.module.state_dict(), "modele.pt")
  print("Checkpoint sauvegardé")

dist.destroy_process_group()

In [ ]:
!torchrun --nproc_per_node=2 entrainement_ddp.py

In [ ]:
import torch

from commun import fabrique_modele

modele_recharge = fabrique_modele()
modele_recharge.load_state_dict(torch.load("modele.pt"))
print("Poids rechargés dans un modèle classique")

`modele.state_dict()` préfixerait toutes les clés par `module.` (le nom de l'attribut dans lequel DDP range le modèle) : ces poids ne pourraient pas être rechargés dans un modèle ordinaire sans renommer les clés.

## Avec PyTorch Lightning

Lightning se charge de tout ce que nous avons écrit à la main : groupe de processus, `DDP`, `DistributedSampler`, `set_epoch`, affichage et sauvegarde depuis le rang 0. Il suffit de choisir une stratégie dans le `Trainer`.

*Complétez le `Trainer` de `entrainement_lightning.py` pour entraîner sur `nb_processus` processus CPU, le nombre passé au script (`python entrainement_lightning.py 1` pour un seul, 2 sans argument), avec la stratégie `"ddp"`.*

In [ ]:
%%writefile entrainement_lightning.py
import sys

import lightning
import torch
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from commun import fabrique_donnees, fabrique_modele


class Classifieur(lightning.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.modele = fabrique_modele()

  def training_step(self, batch, indice_lot):
    x, y = batch
    perte = cross_entropy(self.modele(x), y)
    self.log("perte_entrainement", perte, prog_bar=True)
    return perte

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  # Le nombre passé au script, 2 sans argument
  nb_processus = int(sys.argv[1]) if len(sys.argv) > 1 else 2
  chargeur = DataLoader(fabrique_donnees(), batch_size=64, shuffle=True)
  trainer = lightning.Trainer(
    max_epochs=3,
    ...,  # Votre code ici
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    # Ni checkpoint ni journal : ce TP n'en relit aucun
    enable_checkpointing=False,
    logger=False,
  )
  trainer.fit(Classifieur(), chargeur)
  if trainer.is_global_zero:
    print(f"{trainer.num_training_batches} pas par epoch et par processus")

### Conseils supplémentaires

- `accelerator="cpu"` : des processus sur CPU, comme notre script DDP.
- `devices=nb_processus` : le nombre de processus que Lightning lance, 2 quand le script est lancé sans argument.
- `strategy="ddp"` : un `DistributedDataParallel` par processus, avec le backend `gloo` sur CPU.

```python
trainer = lightning.Trainer(
  max_epochs=3,
  accelerator=...,
  devices=...,
  strategy=...,
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
  logger=False,
)
```

### Solution

In [ ]:
%%writefile entrainement_lightning.py
import sys

import lightning
import torch
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from commun import fabrique_donnees, fabrique_modele


class Classifieur(lightning.LightningModule):
  def __init__(self) -> None:
    super().__init__()
    self.modele = fabrique_modele()

  def training_step(self, batch, indice_lot):
    x, y = batch
    perte = cross_entropy(self.modele(x), y)
    self.log("perte_entrainement", perte, prog_bar=True)
    return perte

  def configure_optimizers(self):
    return torch.optim.SGD(self.parameters(), lr=0.1)


if __name__ == "__main__":
  # Le nombre passé au script, 2 sans argument
  nb_processus = int(sys.argv[1]) if len(sys.argv) > 1 else 2
  chargeur = DataLoader(fabrique_donnees(), batch_size=64, shuffle=True)
  trainer = lightning.Trainer(
    max_epochs=3,
    accelerator="cpu",
    devices=nb_processus,
    strategy="ddp",
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    # Ni checkpoint ni journal : ce TP n'en relit aucun
    enable_checkpointing=False,
    logger=False,
  )
  trainer.fit(Classifieur(), chargeur)
  if trainer.is_global_zero:
    print(f"{trainer.num_training_batches} pas par epoch et par processus")

### Sur un puis deux processus

Lancez le script sur 1 processus, puis sans argument, sur 2. À la fin, il affiche le nombre de pas (de batchs) que chaque processus fait par epoch.

In [ ]:
!python entrainement_lightning.py 1

In [ ]:
!python entrainement_lightning.py

1. *Combien de pas par epoch chaque processus fait-il, sur 1 puis sur 2 processus ? Pourquoi, alors que le `DataLoader` est créé avec `shuffle=True` et sans `DistributedSampler` ?*
2. *Sur une machine avec 4 GPUs, que faudrait-il changer dans le `Trainer` ?*

*Vos réponses ici.*

#### Solution

1. 128 pas par epoch sur 1 processus (8 192 exemples en batchs de 64), 64 sur 2 : avec la stratégie `"ddp"`, Lightning remplace le `shuffle=True` du `DataLoader` par un `DistributedSampler`, dont il appelle lui-même `set_epoch`, et chaque processus ne voit que la moitié des données, comme dans notre script DDP.
2. `accelerator="gpu", devices=4` suffisent (et `precision="bf16-mixed"` pour la précision mixte) : Lightning choisit le backend `nccl` et place chaque processus sur son GPU (`LOCAL_RANK`, `torch.cuda.set_device`).

## Précision mixte (facultatif)

Cette dernière partie, facultative, revient au script sur un seul processus du début, `entrainement_simple.py`. La précision mixte (diapositive « Précision mixte » du cours) calcule la passe avant en 16 bits, mais garde les poids en 32 bits ; elle se combine avec DDP sans rien y changer (dans Lightning, `precision="bf16-mixed"`). Sur GPU, elle accélère nettement l'entraînement ; sur CPU, seulement sur les processeurs qui ont des instructions bfloat16 : ici, nous regardons les types des tenseurs, pas le temps.

*Modifiez `entrainement_simple.py` ci-dessous pour calculer la sortie du modèle et la perte en bfloat16 avec `torch.autocast`, sur CPU. À la fin de l'entraînement, affichez le type de la sortie du modèle, de la perte, des poids de la première couche et de leurs gradients.*

In [ ]:
%%writefile entrainement_simple.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from commun import fabrique_donnees, fabrique_modele

donnees = fabrique_donnees()
modele = fabrique_modele()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
chargeur = DataLoader(donnees, batch_size=64, shuffle=True)

for epoque in range(3):
  for x, y in chargeur:
    optimiseur.zero_grad()
    sortie = modele(x)  # La sortie et la perte, en bfloat16
    perte = cross_entropy(sortie, y)
    perte.backward()
    optimiseur.step()
  print(f"Epoch {epoque} : {len(chargeur)} batchs, dernière perte {perte.item():.3f}")

...  # Types de la sortie, de la perte, des poids et de leurs gradients

### Conseils supplémentaires

- `with torch.autocast(device_type="cpu", dtype=torch.bfloat16):` englobe la passe avant et la perte ; `perte.backward()` et `optimiseur.step()` restent hors du bloc, comme sur la diapositive « Précision mixte ».
- `modele` est un `nn.Sequential` : `modele[0]` est sa première couche, `modele[0].weight` ses poids, et `modele[0].weight.grad` leurs gradients, que `backward()` remplit.
- Le type d'un tenseur : son attribut `.dtype`.

```python
for epoque in range(3):
  for x, y in chargeur:
    optimiseur.zero_grad()
    with torch.autocast(device_type=..., dtype=...):
      sortie = modele(x)
      perte = ...
    perte.backward()
    optimiseur.step()
  print(...)  # Inchangé

print(f"Sortie du modèle : {sortie.dtype}")
print(f"Perte : {...}")
print(f"Poids de la première couche : {...}")
print(f"Gradients de ces poids : {...}")
```

### Solution

In [ ]:
%%writefile entrainement_simple.py
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader

from commun import fabrique_donnees, fabrique_modele

donnees = fabrique_donnees()
modele = fabrique_modele()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)
chargeur = DataLoader(donnees, batch_size=64, shuffle=True)

for epoque in range(3):
  for x, y in chargeur:
    optimiseur.zero_grad()
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
      sortie = modele(x)
      perte = cross_entropy(sortie, y)
    perte.backward()
    optimiseur.step()
  print(f"Epoch {epoque} : {len(chargeur)} batchs, dernière perte {perte.item():.3f}")

print(f"Sortie du modèle : {sortie.dtype}")
print(f"Perte : {perte.dtype}")
print(f"Poids de la première couche : {modele[0].weight.dtype}")
print(f"Gradients de ces poids : {modele[0].weight.grad.dtype}")

### Les types en précision mixte

Lancez le script, qui s'entraîne maintenant en précision mixte :

In [ ]:
!python entrainement_simple.py

1. *Quel type a la sortie du modèle, et quel type a la perte ? Pourquoi ne sont-ils pas les mêmes ?*
2. *Les poids et leurs gradients sont-ils passés en bfloat16 ? Pourquoi est-ce préférable ?*
3. *Avec `dtype=torch.float16`, il faudrait un `torch.amp.GradScaler`. Pourquoi n'en faut-il pas avec bfloat16 ?*

*Vos réponses ici.*

#### Solution

1. La sortie est en `bfloat16` : dans le bloc, autocast calcule les produits matriciels des couches `nn.Linear` en bfloat16, et `ReLU` garde le type de son entrée. La perte est en `float32` : `cross_entropy` fait partie des opérations qu'autocast calcule en float32, parce que ses exponentielles, ses logarithmes et ses sommes perdent trop de précision en 16 bits.
2. Non : autocast ne convertit pas les paramètres, il calcule chaque produit matriciel avec une copie en bfloat16 des poids. Les poids restent en float32, et leurs gradients, rangés dans le `.grad` de chaque paramètre, aussi. C'est voulu : une mise à jour (learning rate × gradient) est souvent très petite devant le poids, et bfloat16 n'a qu'environ 3 chiffres significatifs. En bfloat16, `1 + 0.001` vaut `1` : la mise à jour serait perdue.
3. bfloat16 garde les 8 bits d'exposant de float32, donc la même plage de valeurs : il perd des chiffres significatifs, mais un petit gradient ne devient pas nul. float16 n'a que 5 bits d'exposant : les très petits gradients s'y arrondissent à zéro. `torch.amp.GradScaler` les en protège, en multipliant la perte avant `backward()`, puis en divisant les gradients avant la mise à jour.

## Pour aller plus loin

- Si le modèle ne tient pas dans la mémoire d'un GPU, DDP ne suffit plus : la stratégie `"fsdp"` (*Fully Sharded Data Parallel*) répartit aussi les poids, les gradients et les états de l'optimiseur entre les GPUs.
- Sur plusieurs machines, la même commande se lance sur chacune, avec le nombre de machines et l'adresse de l'une d'elles, où les processus se donnent rendez-vous : `torchrun --nnodes=2 --nproc_per_node=4 --rdzv_id=tp_ddp --rdzv_backend=c10d --rdzv_endpoint=machine0:29500 entrainement_ddp.py`.
- Voir le [tutoriel DDP de PyTorch](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html) et la [documentation de `torchrun`](https://docs.pytorch.org/docs/stable/elastic/run.html).